# Camera positions around the insect

Interactive 3D view of a photo session: every matched photo's camera position
(from `frames.csv`) as a dot **in the insect's body frame**, so the insect
sits at the origin. With the turntable method (fixed camera, insect rotated
between shots) the rotation shows up here as a ring of cameras — exactly the
orbit the reconstruction sees. Drag to rotate, scroll to zoom, hover a dot
for the photo it corresponds to.

**Kernel:** select the pixi env interpreter (`.pixi/envs/default/bin/python`).

Set `SESSION` to the session you want to look at (its `frames.csv` must be
produced by `pixi run match`, which outputs insect-frame poses by default).

In [ ]:
import csv
from pathlib import Path

import numpy as np

SESSION = Path('../datasets/phone_20260708_105106')   # ← change per session
if not SESSION.exists():                               # also works from repo root
    SESSION = Path(str(SESSION).removeprefix('../'))

rows = list(csv.DictReader(open(SESSION / 'frames.csv')))
cams = np.array([[float(r['x']), float(r['y']), float(r['z'])] for r in rows])
names = [Path(r['image_file']).name for r in rows]
gap_ms = [float(r['gap_ms']) for r in rows]
spread_mm = [float(r['pos_spread_mm']) for r in rows]

d = np.linalg.norm(cams, axis=1)
print(f'{len(rows)} camera positions from {SESSION / "frames.csv"}')
print(f'camera–insect distance: min {d.min():.3f} m · '
      f'median {np.median(d):.3f} m · max {d.max():.3f} m')

In [ ]:
import plotly.graph_objects as go

# sequential blue ramp (ordinal steps 250→700) — dot color = shot order
BLUES = ['#86b6ef', '#5598e7', '#3987e5', '#2a78d6', '#256abf',
         '#1c5cab', '#184f95', '#104281', '#0d366b']
colorscale = [[i / (len(BLUES) - 1), c] for i, c in enumerate(BLUES)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(                       # faint path in shot order
    x=cams[:, 0], y=cams[:, 1], z=cams[:, 2], mode='lines',
    line=dict(color='#b5b3ad', width=2), name='shot order', hoverinfo='skip'))
fig.add_trace(go.Scatter3d(
    x=cams[:, 0], y=cams[:, 1], z=cams[:, 2], mode='markers', name='cameras',
    marker=dict(size=8, color=np.arange(len(cams)), colorscale=colorscale,
                colorbar=dict(title='shot order', thickness=12, len=0.6)),
    customdata=[[n, g, s] for n, g, s in zip(names, gap_ms, spread_mm)],
    hovertemplate=('<b>%{customdata[0]}</b><br>'
                   'x=%{x:.3f}  y=%{y:.3f}  z=%{z:.3f} m<br>'
                   'pose gap %{customdata[1]} ms · '
                   'motion ±50 ms %{customdata[2]} mm<extra></extra>')))
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers+text',
    marker=dict(size=10, color='#e34948', symbol='diamond'),
    text=['insect'], textposition='top center',
    textfont=dict(color='#55524c', size=13), name='insect (origin)',
    hovertemplate='insect — body-frame origin<extra></extra>'))
fig.update_layout(
    title=f'Cameras in the insect body frame — {SESSION.name}',
    scene=dict(xaxis_title='x (m)', yaxis_title='y (m)', zaxis_title='z (m)',
               aspectmode='data'),                 # true metric proportions
    template='plotly_white', height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()